In [6]:
!pip install nina_funcs

In [7]:
import os
import glob
import numpy as np
from scipy.io import savemat
import pywt
import numpy as np
import pandas as pd
# Import functions from your ninapro preprocessing library.
from nina_funcs import (
    get_data,
    normalise,
    filter_data,
    notch_filter,
    windowing,
    get_categorical,
    rectify
)

In [8]:
def process_subject_file(file_path, train_reps, test_reps, gestures, win_len, win_stride):
    """
    Process a single subject file.
    
    Parameters:
      file_path (str): Full path to the subject's .mat file.
      train_reps (list): List of training repetitions.
      test_reps (list): List of testing repetitions.
      gestures (list): List of gesture labels to classify.
      win_len (int): Window length (adjusted for sampling rate).
      win_stride (int): Window stride (adjusted for sampling rate).
    
    Returns:
      X_train, y_train_cat, X_test, y_test_cat: Processed data ready for the model.
    """
    data_path = os.path.dirname(file_path)
    file_name = os.path.basename(file_path)
    
    # Load raw data.
    data = get_data(data_path, file_name)

    # Normalize using only training repetitions.
    data = normalise(data, train_reps)
    

    # Segment the data into windows.
    X_train, y_train, _ = windowing(data, train_reps, gestures, win_len, win_stride)
    X_test, y_test, _   = windowing(data, test_reps, gestures, win_len, win_stride)
    
    # Convert labels to one-hot encoding.
    y_train_cat = get_categorical(y_train)
    y_test_cat = get_categorical(y_test)
    
    return X_train, y_train_cat, X_test, y_test_cat

In [ ]:

def main():
    # Define your input and output directories.
    input_folder = fr'D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\dataset'
    output_folder = fr'D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\v3\normalized'
    os.makedirs(output_folder, exist_ok=True)
    
    # Find subject files matching pattern (e.g. S1_E1_A1.mat, S2_E1_A1.mat, etc.).
    subject_files = glob.glob(os.path.join(input_folder, 'S*_E1_A1.mat'))
    if not subject_files:
        print("No subject files found in the specified directory.")
        return
    
    # Define processing parameters.
    gestures = [ 1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 13, 14, 15, 16, 17]
    train_reps = [1, 2, 4, 6]
    test_reps = [3, 5]
    # For a 300ms window at 2000Hz: win_len = 300ms * 2 = 600; for a 10ms stride: win_stride = 10ms * 2 = 20.
    win_len = 300
    win_stride = 200
    
    # Process each subject file.
    for file_path in subject_files:
        print(f"Processing file: {file_path}")
        X_train, y_train_cat, X_test, y_test_cat = process_subject_file(
            file_path, train_reps, test_reps, gestures, win_len, win_stride
        )
        
        # Build output filename.
        base_name = os.path.splitext(os.path.basename(file_path))[0]  # e.g., "S1_E1_A1"
        # Extract subject identifier (assumed to be like "S1") and construct new name.
        subject_id = base_name.split('_')[0]  # "S1"
        output_filename = f"{subject_id}_E1_A1_300_200_N.mat"
        output_filepath = os.path.join(output_folder, output_filename)
        
        # Create dictionary with keys expected by your model code.
        out_data = {
            'train_data': X_train,
            'train_labels': y_train_cat,
            'test_data': X_test,
            'test_labels': y_test_cat
        }
        
        savemat(output_filepath, out_data)
        print(f"Saved preprocessed data to: {output_filepath}")

if __name__ == "__main__":
    main()

Processing file: D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\dataset\S10_E1_A1.mat
Saved preprocessed data to: D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\v3\normalized\S10_E1_A1_300_200_N.mat
Processing file: D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\dataset\S11_E1_A1.mat
Saved preprocessed data to: D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\v3\normalized\S11_E1_A1_300_200_N.mat
Processing file: D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\dataset\S12_E1_A1.mat
Saved preprocessed data to: D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\v3\normalized\S12_E1_A1_300_200_N.mat
Processing file: D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\dataset\S13_E1_A1.mat
Saved preprocessed data to: D: